
# 🧠 Enhanced Member B Notebook — Fake News & Spam Detection

Complete Colab-ready pipeline for **Feature Engineering** and **Classical Machine Learning**
using your cleaned *SMS Spam* and *Fake News* datasets.


In [1]:

!pip install -q xgboost lime shap joblib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive not mounted:', e)

import os, joblib, time, warnings, re
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
print("✅ Setup complete.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Mounted at /content/drive
✅ Setup complete.


In [4]:

DATA_PATHS = {
    'sms': '/content/drive/MyDrive/clean_sms_spam.csv',
    'news_parts': [
        '/content/drive/MyDrive/clean_fake_news.csv'
    ]
}

def try_load(path):
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("❌ ", path, e)
        return None

df_sms  = try_load(DATA_PATHS['sms'])
news_dfs = [d for d in [try_load(p) for p in DATA_PATHS['news_parts']] if d is not None]
df_news = pd.concat(news_dfs, ignore_index=True)
print("📊 Loaded:", df_sms.shape, " (SMS), ", df_news.shape, " (News)")


📊 Loaded: (5565, 3)  (SMS),  (44182, 6)  (News)


In [8]:
def detect_cols(df):
    """
    Automatically detect the text and label columns in a dataframe.
    Works for both SMS spam and Fake News datasets.
    """
    # Identify object (text-like) columns
    obj_cols = [c for c in df.columns if df[c].dtype == 'object']

    if not obj_cols:
        raise ValueError("No text-like columns found in dataframe.")

    # Choose the column with the highest average string length as text column
    avg_len = {c: df[c].astype(str).map(len).mean() for c in obj_cols}
    text_col = max(avg_len, key=avg_len.get)

    # Prefer known label column names
    possible_labels = ['label', 'class', 'target', 'is_fake', 'truth', 'type', 'category']
    for cand in possible_labels:
        if cand in df.columns:
            return text_col, cand

    # Otherwise, pick another non-text column with few unique values as label
    candidates = [c for c in df.columns if c != text_col and df[c].nunique() <= 20]
    label_col = candidates[0] if candidates else [c for c in df.columns if c != text_col][0]

    return text_col, label_col


sms_text, sms_label = detect_cols(df_sms)
news_text, news_label = detect_cols(df_news)
print(f"Detected SMS: text='{sms_text}', label='{sms_label}'")
print(f"Detected News: text='{news_text}', label='{news_label}'")


def map_binary(df, label_col):
    """
    Converts categorical labels to binary (0/1).
    Handles spam/ham, fake/real, and numeric labels automatically.
    """
    uniq = df[label_col].dropna().unique().tolist()

    # If already numeric binary
    if set(uniq).issubset({0, 1}):
        df['label_binary'] = df[label_col].astype(int)
        return df

    low = [str(u).lower() for u in uniq]

    # Common label mappings
    if 'spam' in low and 'ham' in low:
        m = {uniq[low.index('ham')]: 0, uniq[low.index('spam')]: 1}
    elif 'fake' in low or 'false' in low:
        m = {u: (1 if any(k in str(u).lower() for k in ['fake', 'false']) else 0) for u in uniq}
    elif 'real' in low or 'true' in low:
        m = {u: (0 if any(k in str(u).lower() for k in ['real', 'true']) else 1) for u in uniq}
    else:
        # Default mapping based on frequency order
        counts = df[label_col].value_counts()
        vals = counts.index.tolist()
        m = {vals[0]: 0, vals[1]: 1}

    df['label_binary'] = df[label_col].map(m).astype(int)
    return df


# Apply mapping
df_sms = map_binary(df_sms, sms_label)
df_news = map_binary(df_news, news_label)

# Quick verification
print("\n✅ Label mapping successful:")
print("SMS label distribution:\n", df_sms['label_binary'].value_counts())
print("News label distribution:\n", df_news['label_binary'].value_counts())


Detected SMS: text='text', label='label'
Detected News: text='text', label='label'

✅ Label mapping successful:
SMS label distribution:
 label_binary
0    4818
1     747
Name: count, dtype: int64
News label distribution:
 label_binary
1    22766
0    21416
Name: count, dtype: int64


In [9]:

def basic_feats(s):
    s = s.fillna('').astype(str)
    return pd.DataFrame({
        'char_len': s.map(len),
        'word_count': s.map(lambda x: len(x.split())),
        'exclaim': s.map(lambda x: x.count('!')),
        'question': s.map(lambda x: x.count('?')),
        'caps': s.map(lambda x: sum(1 for w in x.split() if w.isupper())),
        'urls': s.map(lambda x: len(re.findall(r'http', x)))
    })

def build_tfidf(train, test, max_features=8000):
    v = TfidfVectorizer(ngram_range=(1,2), max_features=max_features)
    return v, v.fit_transform(train), v.transform(test)

def combine(Xtr_tfidf, Xte_tfidf, ntr, nte):
    sc = StandardScaler()
    ntr_s, nte_s = sc.fit_transform(ntr), sc.transform(nte)
    return hstack([Xtr_tfidf, csr_matrix(ntr_s)]).tocsr(),            hstack([Xte_tfidf, csr_matrix(nte_s)]).tocsr(), sc


In [10]:

Xs, ys = df_sms[sms_text], df_sms['label_binary']
Xn, yn = df_news[news_text], df_news['label_binary']

Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, test_size=0.2, stratify=ys, random_state=42)
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.2, stratify=yn, random_state=42)

v_s, Xs_tr_tfidf, Xs_te_tfidf = build_tfidf(Xs_tr, Xs_te, 5000)
v_n, Xn_tr_tfidf, Xn_te_tfidf = build_tfidf(Xn_tr, Xn_te, 8000)

ns_tr, ns_te = basic_feats(Xs_tr), basic_feats(Xs_te)
nn_tr, nn_te = basic_feats(Xn_tr), basic_feats(Xn_te)
Xs_tr_c, Xs_te_c, sc_s = combine(Xs_tr_tfidf, Xs_te_tfidf, ns_tr, ns_te)
Xn_tr_c, Xn_te_c, sc_n = combine(Xn_tr_tfidf, Xn_te_tfidf, nn_tr, nn_te)
print("✅ Feature shapes:", Xs_tr_c.shape, Xn_tr_c.shape)


✅ Feature shapes: (4452, 5006) (35345, 8006)


In [14]:
from sklearn.preprocessing import MinMaxScaler

# Prepare non-negative feature versions for Naive Bayes
mms = MinMaxScaler()

# SMS dataset
Xs_tr_nb = hstack([Xs_tr_tfidf, csr_matrix(mms.fit_transform(ns_tr))]).tocsr()
Xs_te_nb = hstack([Xs_te_tfidf, csr_matrix(mms.transform(ns_te))]).tocsr()

# Fake News dataset
Xn_tr_nb = hstack([Xn_tr_tfidf, csr_matrix(mms.fit_transform(nn_tr))]).tocsr()
Xn_te_nb = hstack([Xn_te_tfidf, csr_matrix(mms.transform(nn_te))]).tocsr()

print("✅ NB-safe features created (no negative values).")


✅ NB-safe features created (no negative values).


In [15]:
def evaluate(model, Xtr, Xte, ytr, yte, name):
    t0 = time.time()
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    f1 = f1_score(yte, pred, zero_division=0)
    print(f"\n{name} — F1 = {f1:.4f}  |  Time: {time.time()-t0:.1f}s")
    print(classification_report(yte, pred, digits=4))
    return model, f1

models = {
    "Naive Bayes": MultinomialNB(),
    "LogReg": LogisticRegression(max_iter=1000, solver='liblinear'),
    "SVM": LinearSVC(max_iter=5000),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss')
}

results = {"SMS": {}, "News": {}}

print("📡 SMS Models:")
for n, m in models.items():
    if n == "Naive Bayes":
        model, f1 = evaluate(m, Xs_tr_nb, Xs_te_nb, ys_tr, ys_te, f"SMS → {n}")
    else:
        model, f1 = evaluate(m, Xs_tr_c, Xs_te_c, ys_tr, ys_te, f"SMS → {n}")
    results["SMS"][n] = f1

print("\n📰 Fake News Models:")
for n, m in models.items():
    if n == "Naive Bayes":
        model, f1 = evaluate(m, Xn_tr_nb, Xn_te_nb, yn_tr, yn_te, f"News → {n}")
    else:
        model, f1 = evaluate(m, Xn_tr_c, Xn_te_c, yn_tr, yn_te, f"News → {n}")
    results["News"][n] = f1

# Display summary
print("\n📊 Summary (F1-scores):")
for dataset, scores in results.items():
    print(f"\n{dataset} Results:")
    for model, f1 in scores.items():
        print(f"  {model:<15} → F1 = {f1:.4f}")


📡 SMS Models:

SMS → Naive Bayes — F1 = 0.8922  |  Time: 0.0s
              precision    recall  f1-score   support

           0     0.9708    1.0000    0.9852       964
           1     1.0000    0.8054    0.8922       149

    accuracy                         0.9739      1113
   macro avg     0.9854    0.9027    0.9387      1113
weighted avg     0.9747    0.9739    0.9727      1113


SMS → LogReg — F1 = 0.8889  |  Time: 0.1s
              precision    recall  f1-score   support

           0     0.9746    0.9938    0.9841       964
           1     0.9538    0.8322    0.8889       149

    accuracy                         0.9721      1113
   macro avg     0.9642    0.9130    0.9365      1113
weighted avg     0.9718    0.9721    0.9713      1113


SMS → SVM — F1 = 0.9510  |  Time: 1.2s
              precision    recall  f1-score   support

           0     0.9867    0.9990    0.9928       964
           1     0.9927    0.9128    0.9510       149

    accuracy                         

In [17]:
print("\n🔁 Cross-Domain Testing (with numeric features)")

# ---- SMS → News ----
# 1. Use SMS vectorizer & scaler
Xnews_te_on_sms_tfidf = v_s.transform(Xn_te)
num_news_te_for_sms = basic_feats(Xn_te)
num_news_te_scaled_sms = csr_matrix(sc_s.transform(num_news_te_for_sms))
Xnews_te_on_sms_comb = hstack([Xnews_te_on_sms_tfidf, num_news_te_scaled_sms]).tocsr()

# 2. Train on SMS full feature set
svm_s = LinearSVC(max_iter=5000)
svm_s.fit(Xs_tr_c, ys_tr)

# 3. Predict on News (converted into SMS feature space)
pred_news_from_sms = svm_s.predict(Xnews_te_on_sms_comb)
print("SMS → News F1:", f1_score(yn_te, pred_news_from_sms, zero_division=0))


# ---- News → SMS ----
# 1. Use News vectorizer & scaler
Xsms_te_on_news_tfidf = v_n.transform(Xs_te)
num_sms_te_for_news = basic_feats(Xs_te)
num_sms_te_scaled_news = csr_matrix(sc_n.transform(num_sms_te_for_news))
Xsms_te_on_news_comb = hstack([Xsms_te_on_news_tfidf, num_sms_te_scaled_news]).tocsr()

# 2. Train on News full feature set
svm_n = LinearSVC(max_iter=5000)
svm_n.fit(Xn_tr_c, yn_tr)

# 3. Predict on SMS (converted into News feature space)
pred_sms_from_news = svm_n.predict(Xsms_te_on_news_comb)
print("News → SMS F1:", f1_score(ys_te, pred_sms_from_news, zero_division=0))



🔁 Cross-Domain Testing (with numeric features)
SMS → News F1: 0.6605194406024282
News → SMS F1: 0.2368


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from scipy.stats import uniform
import joblib

pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, solver='liblinear'))
])

# Randomized search instead of full grid
params = {
    'tfidf__ngram_range': [(1,1), (1,2)],
    'tfidf__max_features': [5000, 10000],
    'clf__C': uniform(0.1, 5)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rs = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=params,
    n_iter=5,          # test only 5 random combinations
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

print("🚀 Running RandomizedSearch (approx 2–3 min)...")
rs.fit(Xn_tr.astype(str), yn_tr)

print("\n🏁 Best Parameters:", rs.best_params_)
print("✅ Best Cross-Validated F1:", rs.best_score_)

y_pred = rs.predict(Xn_te.astype(str))
print("\n📊 Test Set Performance:")
print(classification_report(yn_te, y_pred, digits=4))
print("Test F1 Score:", f1_score(yn_te, y_pred, zero_division=0))

🚀 Running RandomizedSearch (approx 2–3 min)...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

🏁 Best Parameters: {'clf__C': np.float64(3.7599697090570254), 'tfidf__max_features': 5000, 'tfidf__ngram_range': (1, 1)}
✅ Best Cross-Validated F1: 0.9923072769933322

📊 Test Set Performance:
              precision    recall  f1-score   support

           0     0.9937    0.9965    0.9951      4283
           1     0.9967    0.9941    0.9954      4554

    accuracy                         0.9952      8837
   macro avg     0.9952    0.9953    0.9952      8837
weighted avg     0.9953    0.9952    0.9952      8837

Test F1 Score: 0.9953825857519789


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from scipy.stats import uniform
import joblib

# ✅ Define the pipeline
pipe_sms = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, solver='liblinear'))
])

# ✅ Define the parameter space
params_sms = {
    'tfidf__ngram_range': [(1,1), (1,2)],
    'tfidf__max_features': [3000, 5000],   # smaller since SMS data is short
    'clf__C': uniform(0.1, 5)
}

# ✅ Use StratifiedKFold
cv_sms = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ✅ Randomized Search setup
rs_sms = RandomizedSearchCV(
    estimator=pipe_sms,
    param_distributions=params_sms,
    n_iter=5,
    scoring='f1',
    cv=cv_sms,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

print("🚀 Running RandomizedSearch on SMS Spam dataset (≈2–3 min)...")
rs_sms.fit(Xs_tr.astype(str), ys_tr)

# ✅ Best Parameters and CV Score
print("\n🏁 Best Parameters:", rs_sms.best_params_)
print("✅ Best Cross-Validated F1:", rs_sms.best_score_)

# ✅ Evaluate on test set
y_pred_sms = rs_sms.predict(Xs_te.astype(str))
print("\n📊 Test Set Performance:")
print(classification_report(ys_te, y_pred_sms, digits=4))
print("Test F1 Score:", f1_score(ys_te, y_pred_sms, zero_division=0))

# ✅ Save model
joblib.dump(rs_sms.best_estimator_, '/content/drive/MyDrive/best_sms_spam_model.joblib')
print("\n💾 SMS Spam model saved to Drive.")


🚀 Running RandomizedSearch on SMS Spam dataset (≈2–3 min)...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

🏁 Best Parameters: {'clf__C': np.float64(3.7599697090570254), 'tfidf__max_features': 3000, 'tfidf__ngram_range': (1, 1)}
✅ Best Cross-Validated F1: 0.9160151770427228

📊 Test Set Performance:
              precision    recall  f1-score   support

           0     0.9806    0.9969    0.9887       964
           1     0.9774    0.8725    0.9220       149

    accuracy                         0.9802      1113
   macro avg     0.9790    0.9347    0.9553      1113
weighted avg     0.9802    0.9802    0.9798      1113

Test F1 Score: 0.9219858156028369

💾 SMS Spam model saved to Drive.


In [23]:
from lime.lime_text import LimeTextExplainer
from sklearn.pipeline import make_pipeline
lr_demo = LogisticRegression(max_iter=1000, solver='liblinear').fit(Xn_tr_tfidf, yn_tr)
pipe_lr = make_pipeline(v_n, lr_demo)
exp = LimeTextExplainer(class_names=['real','fake'])
for txt in list(Xn_te[:2]):
    e = exp.explain_instance(txt, pipe_lr.predict_proba, num_features=6)
    print('\nTEXT:', txt[:200])
    print('LIME:', e.as_list())

import shap
expl = shap.LinearExplainer(lr_demo, v_n.transform(Xn_tr[:200]))
vals = expl.shap_values(v_n.transform(Xn_te[:3]))
fn = v_n.get_feature_names_out()
for i,v in enumerate(vals):
    top = np.argsort(np.abs(v))[-10:][::-1]
    print('\nSample', i, '→', [(fn[j], round(v[j],4)) for j in top])



TEXT: LISBON (Reuters) - Former Portuguese prime minister Jose Socrates was indicted on graft and money laundering charges on Wednesday in a vast corruption investigation that he has dismissed as politicall
LIME: [(np.str_('in'), -0.10063237582540858), (np.str_('said'), -0.07191552513378069), (np.str_('minister'), -0.06940856984259212), (np.str_('Reuters'), -0.06245679733850612), (np.str_('on'), -0.06202555276143264), (np.str_('via'), 0.04492254840734307)]

TEXT: VALLETTA (Reuters) - Thousands of Maltese called for justice on Sunday in a protest held by a group of non-governmental organizations after a journalist was killed last Monday. The demonstrations in M
LIME: [(np.str_('said'), -0.14947769518884937), (np.str_('in'), -0.13875728917138427), (np.str_('the'), 0.11945043159289587), (np.str_('on'), -0.1137263671744256), (np.str_('Reuters'), -0.07846815662239044), (np.str_('of'), -0.05794493072677719)]

Sample 0 → [('minister', np.float64(-0.4829)), ('in', np.float64(-0.4292)), ('via'

In [24]:
# ================================================
# ✅ Confidence Thresholding & Uncertain Prediction
# ================================================

import pandas as pd
import numpy as np

# We'll use Logistic Regression (or any model with predict_proba)
best_model = LogisticRegression(max_iter=1000, solver='liblinear')
best_model.fit(Xn_tr_c, yn_tr)

# Get predicted probabilities on test data
probs = best_model.predict_proba(Xn_te_c)
confidence_scores = probs.max(axis=1)
predicted_labels = best_model.classes_[np.argmax(probs, axis=1)]

# Define confidence threshold
threshold = 0.7  # you can adjust this (0.7–0.8 works well)

final_preds = []
for conf, label in zip(confidence_scores, predicted_labels):
    if conf < threshold:
        final_preds.append("uncertain")
    else:
        final_preds.append(label)

# Combine results for display
df_conf = pd.DataFrame({
    "Text_Sample": Xn_te[:10].values,
    "Predicted_Label": final_preds[:10],
    "Confidence": confidence_scores[:10]
})

print("🔍 Sample predictions with confidence threshold applied:")
print(df_conf)

# Count how many were marked uncertain
uncertain_count = sum(1 for x in final_preds if x == "uncertain")
print(f"\n⚙️  Total uncertain predictions: {uncertain_count} out of {len(final_preds)}")


🔍 Sample predictions with confidence threshold applied:
                                         Text_Sample  Predicted_Label  \
0  LISBON (Reuters) - Former Portuguese prime min...                0   
1  VALLETTA (Reuters) - Thousands of Maltese call...                0   
2  Another Republican has dropped out of the race...                1   
3  GENEVA (Reuters) - The U.N. aid coordinator ca...                0   
4  WASHINGTON (Reuters) - The Senate Finance Comm...                0   
5  NEW DELHI (Reuters) - An Indian court on Thurs...                0   
6  Nobody can accuse Fox News of being sane most ...                1   
7  The #NFL has become a showcase for an anti-law...                1   
8  Democrat dummy of the day Rep. Tom Cole (R., O...                1   
9  Republican Rep. Steve King: The base is going ...                1   

   Confidence  
0    0.972060  
1    0.940900  
2    0.998707  
3    0.986267  
4    0.993226  
5    0.975651  
6    0.998806  
7    0.98336

In [25]:
SAVE_DIR = '/content/drive/MyDrive/Colab Notebooks/memberB_models'
os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(v_n,  os.path.join(SAVE_DIR,'tfidf_news.joblib'))
joblib.dump(v_s,  os.path.join(SAVE_DIR,'tfidf_sms.joblib'))
final = LinearSVC(max_iter=5000).fit(Xn_tr_c, yn_tr)
joblib.dump(final, os.path.join(SAVE_DIR,'lsvc_news.joblib'))
print('💾 Saved artifacts to', SAVE_DIR)


💾 Saved artifacts to /content/drive/MyDrive/Colab Notebooks/memberB_models
